# Ingeniero de boxes para simracing

**Carlos Peñalver · Proyecto del módulo de IA Generativa**

Estado: corpus preparado; agente e indexación pendientes. Las celdas de preparación funcionan sin llamadas a Gemini. El agente todavía no está implementado; las secciones pendientes no constituyen una demo funcional.

Stack: Gemini (LLM y embeddings), ChromaDB, LangGraph y LangChain. Interfaz Streamlit como bonus después del núcleo.

Plan: `../docs/PLAN.md`. Fuentes: `../docs/FUENTES.md`. Evaluación: `../docs/EVALUACION.md`.


## 0. Preparación del entorno

Seleccionar el Python de `.venv` de **Proyecto**, no el entorno de los notebooks de clase. Instalar antes `requirements.txt` siguiendo el README.

Esta celda localiza la raíz desde `Proyecto/` o `Proyecto/notebooks/`. Solo carga el `.env` de este proyecto y no muestra las claves. Los modelos se elegirán antes de la primera prueba de API.


In [1]:
from pathlib import Path
from importlib.metadata import version
import os
import sys

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "requirements.txt").is_file() or not (ROOT / "prompts/system.md").is_file():
    raise RuntimeError("Abre el notebook desde Proyecto o Proyecto/notebooks.")

load_dotenv(ROOT / ".env", override=False)
CORPUS_DIR = ROOT / "data/corpus"
CHROMA_DIR = ROOT / "data/chroma"
SYSTEM_PROMPT = (ROOT / "prompts/system.md").read_text(encoding="utf-8")

print("Python:", sys.version.split()[0])
print("Proyecto:", ROOT.name)
for package in ("langchain-google-genai", "langgraph", "chromadb", "pypdf"):
    print(f"{package}: {version(package)}")
for name in ("GOOGLE_API_KEY", "GEMINI_MODEL", "GEMINI_EMBEDDING_MODEL"):
    print(f"{name}: {'configurado' if os.getenv(name, '').strip() else 'pendiente'}")


Python: 3.12.3
Proyecto: Proyecto
langchain-google-genai: 4.4.2
langgraph: 1.2.15
chromadb: 1.5.9
pypdf: 6.20.1
GOOGLE_API_KEY: configurado
GEMINI_MODEL: pendiente
GEMINI_EMBEDDING_MODEL: pendiente


In [2]:
# Comprobación de importaciones: no instancia clientes ni realiza llamadas de API.
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from pypdf import PdfReader

print("Importaciones del stack correctas. No se ha llamado a Gemini.")


Importaciones del stack correctas. No se ha llamado a Gemini.


## 1. Dominio y corpus — preparado

Ingeniero de boxes para iRacing, ACC y LMU. El alumno ha elegido tres fuentes: árbol público de Ultimate Setup Hub (25 diagnósticos), chuleta de Chris Haye en SimRacingCockpit (20 fichas) y Driver61 ACC Setup Guide V1 (30 fichas).

Son **tres fuentes y 75 fichas**, no 75 documentos originales ni 20 páginas artificiales. Inventario activo: `docs/corpus_mvp.json`. Procedencia, preparación y reproducción: `docs/FUENTES.md`. Los originales, transcripciones y JSONL son locales y no están en Git; las seis fuentes anteriores quedan en reserva.

Se revisó la transcripción de las láminas visualmente; no se validaron los ajustes en pista ni su vigencia para todos los coches. La evaluación medirá fidelidad documental, citas y memoria, sin necesitar simulador.


## 2. Carga y segmentación — preparadas; indexación pendiente

La celda siguiente verifica las huellas del inventario, reconstruye el JSONL y carga cada ficha como un `Document` con metadatos. Cada caso completo es un fragmento inicial: mantiene juntos síntoma, fase, velocidad, condiciones y alternativas. No se vuelve a dividir la tabla por caracteres, lo que podría separar una recomendación de su contexto.

No llama a Gemini. Primero deben existir los archivos locales descritos en `docs/FUENTES.md`. Resultado esperado y comprobado: 25 documentos LMU, 20 iRacing y 30 ACC.

**Siguiente fase:** elegir modelos Gemini, generar embeddings y persistir en Chroma; verificar recuperación filtrada por simulador, sin duplicación al indexar de nuevo. Registrar modelo y configuración del índice. No hay todavía un índice ni resultados de recuperación.


In [3]:
from collections import Counter
import runpy
from langchain_core.documents import Document

builder = runpy.run_path(str(ROOT / "scripts/build_corpus.py"))
records = builder["build"](ROOT)
documents = [
    Document(page_content=record["text"], metadata={
        key: value for key, value in record.items() if key != "text"
    })
    for record in records
]
counts = Counter(doc.metadata["simulator"] for doc in documents)
assert counts == {"lmu": 25, "iracing": 20, "acc": 30}
assert len({doc.metadata["id"] for doc in documents}) == 75
print("Fuentes:", len({doc.metadata["source_id"] for doc in documents}))
print("Fichas por simulador:", dict(counts))
print("Fragmentos completos:", len(documents))
print("Sin llamadas a Gemini; indexación pendiente.")


Fuentes: 3
Fichas por simulador: {'lmu': 25, 'iracing': 20, 'acc': 30}
Fragmentos completos: 75
Sin llamadas a Gemini; indexación pendiente.


## 3. System prompt — borrador disponible

El borrador está en `prompts/system.md` y se carga en `SYSTEM_PROMPT`. Su justificación está en el README. Ajustarlo al corpus real antes de usarlo.

Debe delimitar dominio, tono, fuentes, aplicabilidad por simulador/coche/versión, falta de información y tratamiento del historial. Distinguir diagnóstico provisional, prueba propuesta y resultado observado. Mostrar en la entrega el texto final utilizado, sin secretos.


## 4. Agente en LangGraph — pendiente

Implementar recepción de pregunta, recuperación de Chroma y generación con Gemini. Recuperar contexto en **cada consulta** y entregar identificadores de fuente al modelo.

Compartir la lógica con la futura interfaz web. Dibujar o mostrar el grafo final y explicar sus nodos. No afirmar que un reglaje se ha aplicado o ha mejorado el coche si el piloto no lo ha confirmado. Un solo grafo utilizará el contexto de simulador para filtrar la recuperación.


## 5. Memoria — pendiente

Añadir un checkpointer y un identificador de conversación. Usar el historial para interpretar preguntas de seguimiento antes de recuperar documentos.

Demostrar una referencia a una respuesta anterior y comprobar que una conversación nueva no hereda mensajes de otra. Diferenciar cambios recomendados y aplicados; al cambiar de simulador o coche, revisar el contexto y no arrastrar valores anteriores. Documentar que la memoria inicial en RAM se pierde al reiniciar.


## 6. Chat y cinco ejemplos documentados — pendiente

Crear una celda de chat interactivo con una salida clara, por ejemplo `/salir`. Después, registrar al menos cinco preguntas y sus respuestas **reales**, con fuentes y un seguimiento. No sustituir las respuestas por expectativas.

| Caso | Pregunta prevista | Respuesta real y fuentes |
| --- | --- | --- |
| E01 | Diagnóstico inicial sin contexto suficiente | Pendiente de ejecución |
| E02 | iRacing: síntoma y prueba de ajuste documentada | Pendiente de ejecución |
| E03 | ACC: consulta documental sobre pianos | Pendiente de ejecución |
| E04 | LMU: sobreviraje por fase y velocidad | Pendiente de ejecución |
| E05 | Seguimiento de E02: cambio aplicado y nuevo síntoma | Pendiente de ejecución |

Usar el corpus elegido y los casos de docs/EVALUACION.md. Identificar los escenarios ficticios del piloto y añadir casos sin evidencia, setup fijo y cambio de simulador. Las respuestas de Gemini se documentarán tras ejecutarlas; no equivalen a validación en pista.


## 7. Comparación con Gemini sin RAG — pendiente

Seguir `docs/EVALUACION.md`: mismo modelo, configuración y preguntas; guardar respuestas reales de ambas condiciones. Dar a ambas variantes el mismo contexto e historial del piloto; los fragmentos recuperados son la diferencia. Separar conceptos generales y consultas específicas de cada simulador.

Valorar corrección, aplicabilidad, memoria, citas y utilidad. Registrar fallos y límites; no asumir que RAG gana ni que una respuesta demuestra mejoras de tiempo. Si hay tiempo, comparar también pegando el mismo material documental a Gemini.


## 8. Bonus: Streamlit y presentación — pendiente

Crear una interfaz de chat sobre el mismo núcleo, con selector de iRacing/ACC/LMU, contexto de coche y condiciones, historial aislado por sesión y fuentes visibles. Añadir Streamlit a las dependencias cuando se implemente y verificar secretos antes de desplegar.

Preparar demo de 5–10 minutos y comprobar la ejecución del notebook desde un kernel limpio. Revisar que sus salidas no contienen credenciales ni información privada.
